# AoC 2024 Day 13 — Claw Contraption

**Spark — Cramer's rule as a single vectorised projection**

Puzzle: <https://adventofcode.com/2024/day/13>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A list of claw machines, one block each: button **A** moves the claw by some `(X, Y)`, button **B** by another, and the prize sits at a fixed `(X, Y)`. The claw starts at the origin and must land **exactly** on the prize.

An A press costs **3 tokens**, a B press costs **1**. No button needs pressing more than **100** times.

- **Part 1** — some machines cannot be won at all. Add up the token cost of winning every machine that can be.

## The approach

Each machine is a **2×2 linear system**:

```
a·ax + b·bx = px
a·ay + b·by = py
```

Two equations, two unknowns. Unless the buttons are parallel, there is exactly **one** real solution, and Cramer's rule writes it down without iterating:

```
det   = ax·by − ay·bx
a     = (px·by − py·bx) / det
b     = (ax·py − ay·px) / det
```

That matters more than it looks. The puzzle's framing ("no more than 100 presses") invites a nested loop, and the reference implementation takes that bait — 101 candidate A-press counts per machine, checked one at a time. It also invites talk of "cheapest" solution, but with a unique solution there is nothing to minimise: the only valid answer is the only answer.

Cramer's rule collapses all of it into **one projection over the whole input** — no loop, no search, no per-machine control flow. Every machine is a row, every step is a column expression, and the answer is a single `sum`. This is the shape Spark is actually good at: arithmetic that is identical across millions of independent rows.

The filters afterwards are just the puzzle's constraints restated as predicates — `det != 0` (buttons not parallel), both numerators divisible by `det` (whole presses, since you cannot press a button 1.5 times), and both counts within `0..100`.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day13

spark = get_spark('aoc-2024-day13')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = 'Button A: X+94, Y+34\nButton B: X+22, Y+67\nPrize: X=8400, Y=5400\n\nButton A: X+26, Y+66\nButton B: X+67, Y+21\nPrize: X=12748, Y=12176\n\nButton A: X+17, Y+86\nButton B: X+84, Y+37\nPrize: X=7870, Y=6450\n\nButton A: X+69, Y+23\nButton B: X+27, Y+71\nPrize: X=18641, Y=10279\n'

print('part 1:', day13.part1(spark, EXAMPLE), '(expected 480)')

### Cramer, machine by machine

Four rows, four solved systems. Two of them come out fractional — those are the machines the puzzle says can never be won, and `whole_presses` is exactly why.

In [ ]:
from pyspark.sql import functions as F

machines = day13.parse(spark, EXAMPLE)
machines.show()

det = F.col('ax') * F.col('by') - F.col('ay') * F.col('bx')
a_num = F.col('px') * F.col('by') - F.col('py') * F.col('bx')
b_num = F.col('ax') * F.col('py') - F.col('ay') * F.col('px')

# Cramer, then the two reasons a machine is unwinnable, side by side.
machines.select(
    det.alias('det'),
    a_num.alias('a_num'),
    b_num.alias('b_num'),
    (a_num / det).alias('a_exact'),
    (b_num / det).alias('b_exact'),
    ((a_num % det == 0) & (b_num % det == 0)).alias('whole_presses'),
    ((a_num / det).between(0, 100) & (b_num / det).between(0, 100)).alias('in_range'),
    (3 * (a_num / det).cast('long') + (b_num / det).cast('long')).alias('cost_if_won'),
).show()

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 13)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day13.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day13 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- Parsing leans on `regexp_extract_all(block, r"\d+")` returning the six numbers **in source order**: `ax, ay, bx, by, px, py`. It never looks at the labels. That is fine for AoC's rigid block format and would be fragile against anything else.
- The pattern is `\d+`, **not** `-?\d+` — day 13 has no negative numbers, unlike day 14. Reusing day 14's pattern here would still work; reusing this one there would silently drop the minus signs.
- Values are `LONG`. `px · by` stays small for part 1, but the products are the first thing to overflow if the coordinates ever grow.
- **`det == 0` is filtered before the modulo, not after.** Order matters: `x % 0` raises under ANSI mode rather than returning null. AoC inputs contain no parallel-button machines, so the branch never fires, but leaving it out makes correctness depend on the input.
- `(a_num / det)` produces a **double**; the `.cast("long")` afterwards truncates. It is only safe because the divisibility filter already guaranteed an exact integer. Casting first and filtering second would round wrong answers into the total.
- `int(total or 0)` guards the case where every machine is filtered out — `sum` over zero rows is null, not 0.
- The `0..100` bound is a part 1 constraint from the puzzle text, not a property of the maths.